# Naïve Bayes on a Review Classification Dataset

**Objective:** classify customer reviews as positive (`Liked = 1`) or negative (`Liked = 0`) using Bernoulli Naïve Bayes.

This notebook covers dataset exploration, text preprocessing, CountVectorizer and TF-IDF, an 80/20 train-test split, model training, evaluation, and new-review prediction.

**Dataset file:** `Reviews_data.csv` (keep it in the same folder as this notebook).


## 1. Import libraries

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import BernoulliNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)
from nltk.stem import PorterStemmer


## 2. Load the dataset

In [ ]:
df = pd.read_csv("Reviews_data.csv")

# Remove accidental spaces from column names.
df.columns = df.columns.str.strip()
df = df.rename(columns={"Review": "review"})

print("Dataset shape:", df.shape)
display(df.head(10))
print("\\nDataset information:")
df.info()


## 3. Explore the dataset

In [ ]:
print("Samples per class:")
display(df["Liked"].value_counts().rename(index={0: "Negative (0)", 1: "Positive (1)"}))

print("\\nMissing values:")
display(df.isnull().sum())

# Review length is measured in characters.
df["review_length"] = df["review"].astype(str).str.len()

plt.figure(figsize=(8, 4))
sns.histplot(data=df, x="review_length", hue="Liked", bins=30, kde=True)
plt.title("Review Text Length Distribution")
plt.xlabel("Review length (characters)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(5, 4))
sns.countplot(data=df, x="Liked")
plt.title("Number of Reviews per Sentiment Class")
plt.xticks([0, 1], ["Negative", "Positive"])
plt.tight_layout()
plt.show()


## 4. Handle missing values

In [ ]:
df = df.dropna(subset=["review", "Liked"]).copy()
df["review"] = df["review"].astype(str)
df["Liked"] = df["Liked"].astype(int)

print("Dataset shape after removing missing rows:", df.shape)


## 5. Text preprocessing

The function below lowercases text, expands common contractions, removes URLs, numbers, punctuation and extra spaces, removes English stopwords while preserving negation words, and applies Porter stemming.

**Why preserve `not` and `no`?** They can reverse the sentiment. For example, "good" and "not good" should not become the same cleaned text.


In [ ]:
stemmer = PorterStemmer()
stop_words = set(ENGLISH_STOP_WORDS) - {"no", "not", "nor", "never"}

contractions = {
    "didn't": "did not", "don't": "do not", "doesn't": "does not",
    "isn't": "is not", "wasn't": "was not", "weren't": "were not",
    "can't": "can not", "cannot": "can not", "won't": "will not",
    "couldn't": "could not", "shouldn't": "should not",
    "wouldn't": "would not", "haven't": "have not",
    "hasn't": "has not", "hadn't": "had not",
    "aren't": "are not", "it's": "it is", "that's": "that is",
    "they're": "they are", "we're": "we are", "you're": "you are",
    "i'm": "i am"
}

def clean_text(text):
    # Convert to lowercase.
    text = str(text).lower()

    # Expand contractions before removing punctuation.
    for contraction, expanded in contractions.items():
        text = text.replace(contraction, expanded)

    # Remove URLs, numbers, punctuation and special characters.
    text = re.sub(r"https?://\\S+|www\\.\\S+", " ", text)
    text = re.sub(r"\\d+", " ", text)
    text = re.sub(r"[^a-z\\s]", " ", text)

    # Remove extra spaces and split into words.
    words = re.sub(r"\\s+", " ", text).strip().split()

    # Remove stopwords and apply stemming.
    words = [stemmer.stem(word) for word in words if word not in stop_words]
    return " ".join(words)

df["cleaned_review"] = df["review"].apply(clean_text)
display(df[["review", "cleaned_review", "Liked"]].head(10))


## 6. Split the data into training and testing sets

In [ ]:
X = df["cleaned_review"]
y = df["Liked"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


## 7. Train Bernoulli Naïve Bayes models

We compare two text representations:
- **CountVectorizer with `binary=True`:** represents whether each word is present (1) or absent (0).
- **TF-IDF with `binary=True`:** creates binary word-presence features from the TF-IDF vocabulary.

Both models use BernoulliNB, which is designed for binary features. The vectorizers are fitted only on training data.


In [ ]:
count_model = Pipeline([
    ("vectorizer", CountVectorizer(binary=True)),
    ("classifier", BernoulliNB())
])

tfidf_model = Pipeline([
    ("vectorizer", TfidfVectorizer(binary=True)),
    ("classifier", BernoulliNB())
])

count_model.fit(X_train, y_train)
tfidf_model.fit(X_train, y_train)

print("Both BernoulliNB models trained successfully.")


## 8. Evaluate both models

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)

    print(f"\\n===== {name} =====")
    print("Accuracy :", round(accuracy_score(y_test, y_pred), 4))
    print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 4))
    print("Recall   :", round(recall_score(y_test, y_pred, zero_division=0), 4))
    print("F1 Score :", round(f1_score(y_test, y_pred, zero_division=0), 4))

    print("\\nClassification report:")
    print(classification_report(
        y_test, y_pred,
        target_names=["Negative", "Positive"],
        zero_division=0
    ))

    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        display_labels=["Negative", "Positive"],
        cmap="Blues"
    )
    plt.title(f"Confusion Matrix - {name}")
    plt.tight_layout()
    plt.show()

    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0)
    }

results = [
    evaluate_model("CountVectorizer + BernoulliNB", count_model, X_test, y_test),
    evaluate_model("TF-IDF + BernoulliNB", tfidf_model, X_test, y_test)
]

results_df = pd.DataFrame(results).sort_values("F1 Score", ascending=False)
display(results_df)


## 9. Select and save the best model

In [ ]:
import pickle

best_name = results_df.iloc[0]["Model"]
best_model = count_model if best_name.startswith("Count") else tfidf_model

print("Selected model:", best_name)

# Save the trained vectorizer + classifier together as one pipeline.
# The model expects already-cleaned review text.
with open("sentiment_model.pkl", "wb") as file:
    pickle.dump(best_model, file)

print("Model saved as sentiment_model.pkl")


## 10. Predict sentiment for new reviews

In [ ]:
new_reviews = [
    "The food was fantastic!",
    "Worst service ever.",
    "I loved the friendly staff and delicious food.",
    "The meal was cold and the service was terrible."
]

cleaned_reviews = [clean_text(review) for review in new_reviews]
predictions = best_model.predict(cleaned_reviews)

for review, prediction in zip(new_reviews, predictions):
    sentiment = "Positive" if prediction == 1 else "Negative"
    print(f"Review: {review}")
    print(f"Predicted sentiment: {sentiment}\\n")


## 11. Streamlit deployment

The project folder also contains:
- `app.py` – Streamlit interface;
- `preprocessing.py` – the same cleaning function used for new reviews;
- `sentiment_model.pkl` – trained model pipeline;
- `requirements.txt` – required packages;
- `Reviews_data.csv` – dataset.

Run locally from the project folder:

```bash
pip install -r requirements.txt
streamlit run app.py
```

To deploy:
1. Push `app.py`, `preprocessing.py`, `sentiment_model.pkl`, and `requirements.txt` to a GitHub repository.
2. Open https://share.streamlit.io/ and sign in with GitHub.
3. Create an app, select your repository and branch, and set the main file path to `app.py`.
4. Click Deploy.

**Important:** The notebook's `clean_text()` function and `preprocessing.py` use the same cleaning steps. The saved model receives cleaned text, so the Streamlit app must call `clean_text()` before `model.predict()`.


## 12. Conclusion

In this project, we explored and cleaned customer review data, converted text into numerical features using CountVectorizer and TF-IDF, and trained Bernoulli Naïve Bayes models. We evaluated both models using accuracy, precision, recall, F1-score, confusion matrices and classification reports. Finally, we selected the model with the higher F1-score, tested new reviews and prepared a Streamlit application for deployment.

Use the actual metric values printed by your notebook in your final report.
